# Paper Hunter: download papers from a CSV of DOIs

This notebook shows how to:

1. Install Paper Hunter from GitHub with `pip`
2. Load DOIs from a CSV file
3. Download the PDFs
4. Export the results: PDFs, a detailed report (CSV / Excel / JSON / Markdown), a list of failed DOIs, and the ZIP file(s), optionally split into N parts

## 1. Install

Run this, then **restart the kernel** so the freshly installed code is used. Re-run it whenever you want the latest version from GitHub.

In [ ]:
# 1st line installs the dependencies; 2nd line always fetches the latest Paper Hunter code from GitHub
# (pip skips a reinstall when the version number is unchanged).
# Restart the kernel after running this cell, otherwise the previously imported code keeps running.
%pip install -q "paper-hunter[notebook] @ git+https://github.com/rasoulnorouzi/paper_hunter.git"
%pip install -q --force-reinstall --no-deps "paper-hunter @ git+https://github.com/rasoulnorouzi/paper_hunter.git"

## 2. Settings

Edit these values. `CSV_PATH` can be a local file or a URL; the default is the sample CSV in the repository.

The environment variables have to be set **before** `downloader` is imported.

In [ ]:
import os
from pathlib import Path

CSV_PATH = "https://raw.githubusercontent.com/rasoulnorouzi/paper_hunter/main/sample_doi.csv"
DOI_COLUMN = None                   # None = detect the DOI column automatically, or give its name
N_SPLITS = 1                        # number of ZIP files to split the results into
OUTPUT_DIR = Path("paper_hunter_output")
LIMIT = None                        # e.g. 5 for a quick test run; None = all DOIs
USE_BROWSER = True                  # headless Chrome fallback (needs Chrome/Chromium)

os.environ["PAPER_HUNTER_MIN_DELAY"] = "1"   # random pause between DOIs (seconds)
os.environ["PAPER_HUNTER_MAX_DELAY"] = "3"
if not USE_BROWSER:
    os.environ["PAPER_HUNTER_BROWSER"] = "0"

## 3. Load DOIs from the CSV

`load_dois_from_csv` handles the messy parts:

- finds the DOI column by itself (columns with "doi" in the name come first, otherwise every column is scanned for DOI-looking values)
- detects the separator (`,` `;` tab `|`) and CSVs without a header row
- skips empty or whitespace-only rows, non-DOI values and duplicates
- accepts `doi:10.…` and `https://doi.org/10.…` as well as bare DOIs

In [ ]:
import pandas as pd
from downloader import find_doi_column, load_dois_from_csv, read_doi_table

print("Detected DOI column:", find_doi_column(read_doi_table(CSV_PATH)) if DOI_COLUMN is None else DOI_COLUMN)

dois = load_dois_from_csv(CSV_PATH, column=DOI_COLUMN)
if LIMIT:
    dois = dois[:LIMIT]

print(f"{len(dois)} DOI(s) to download")
dois[:5]

## 4. Download

`run_bulk_download` tries Unpaywall → OpenAlex → Crossref → Sci-Hub for each DOI and stops at the first PDF it finds.

The small log handler below records which source delivered each paper, so the report can include it.

In [ ]:
import logging
from downloader import run_bulk_download


class SourceTracker(logging.Handler):
    """Remember which strategy succeeded for each DOI, based on the log messages."""

    def __init__(self):
        super().__init__()
        self.current = None
        self.source = {}

    def emit(self, record):
        msg = record.getMessage()
        if msg.startswith("--- Starting download process for DOI: "):
            self.current = msg.removeprefix("--- Starting download process for DOI: ").removesuffix(" ---")
        elif msg.startswith("SUCCESS with ") and self.current:
            self.source[self.current] = msg.split()[2].rstrip(".").removesuffix("Downloader")


tracker = SourceTracker()
# Google Colab sets the root logger to WARNING; the tracker needs the INFO messages.
logging.getLogger().setLevel(logging.INFO)
logging.getLogger().addHandler(tracker)
try:
    results_df = run_bulk_download(dois, download_dir=OUTPUT_DIR, use_browser=USE_BROWSER)
finally:
    logging.getLogger().removeHandler(tracker)

results_df.head()

## 5. Build the report

In [ ]:
from datetime import datetime
from utility import _sanitize_doi

report = results_df.copy()
report["source"] = report["doi"].map(tracker.source).fillna("")
report["file"] = report["doi"].map(lambda d: f"{_sanitize_doi(d)}.pdf")
report["size_kb"] = [
    round((OUTPUT_DIR / f).stat().st_size / 1024, 1) if ok and (OUTPUT_DIR / f).exists() else None
    for f, ok in zip(report["file"], report["success"])
]
report.loc[~report["success"], "file"] = ""
report["doi_url"] = "https://doi.org/" + report["doi"]

total = len(report)
n_ok = int(report["success"].sum())
summary = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "input_csv": str(CSV_PATH),
    "output_dir": str(OUTPUT_DIR.resolve()),
    "total": total,
    "downloaded": n_ok,
    "failed": total - n_ok,
    "success_rate": round(n_ok / total, 3) if total else 0.0,
    "by_source": report.loc[report["success"], "source"].value_counts().to_dict(),
}

print(f"Downloaded {n_ok}/{total} ({summary['success_rate']:.1%})")
print("By source:", summary["by_source"])
report

## 6. Export the output files

Everything is written to `OUTPUT_DIR`:

| File | Content |
|---|---|
| `*.pdf` | the downloaded papers |
| `download_summary.csv` | the summary written by Paper Hunter (`doi`, `success`) |
| `report.csv` / `report.xlsx` / `report.json` | the detailed per-DOI report |
| `report.md` | a readable summary with the failed DOIs |
| `failed_dois.csv` | DOIs that were not found, in the same format as the input, ready to retry |

In [ ]:
import json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_cols = ["doi", "success", "source", "file", "size_kb", "doi_url"]

report[report_cols].to_csv(OUTPUT_DIR / "report.csv", index=False)

try:
    report[report_cols].to_excel(OUTPUT_DIR / "report.xlsx", index=False)
except ImportError:
    print("openpyxl not installed: skipping report.xlsx")

with open(OUTPUT_DIR / "report.json", "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "papers": report[report_cols].to_dict(orient="records")}, f, indent=2)

failed = report.loc[~report["success"], ["doi"]]
failed.to_csv(OUTPUT_DIR / "failed_dois.csv", index=False)

lines = [
    "# Paper Hunter report",
    "",
    f"- Generated: {summary['generated_at']}",
    f"- Input: `{summary['input_csv']}`",
    f"- Downloaded: **{n_ok}/{total}** ({summary['success_rate']:.1%})",
    f"- Failed: **{total - n_ok}**",
    "",
    "## Papers by source",
    "",
    *[f"- {src}: {n}" for src, n in summary["by_source"].items()],
    "",
    "## Failed DOIs",
    "",
    *([f"- [{d}](https://doi.org/{d})" for d in failed["doi"]] or ["None"]),
    "",
]
(OUTPUT_DIR / "report.md").write_text("\n".join(lines), encoding="utf-8")

sorted(p.name for p in OUTPUT_DIR.iterdir() if p.suffix != ".pdf")

### Zip the results (optionally split into N parts)

`zip_downloads_split` divides the DOIs, in input order, into `N_SPLITS` parts of about equal size, e.g. 100 DOIs with `N_SPLITS = 5` gives 5 ZIPs of 20 DOIs each. Each ZIP holds the PDFs of its part and a `download_summary.csv` for exactly those DOIs; the full report files are added to every part.

In [ ]:
from downloader import zip_downloads_split

report_files = [OUTPUT_DIR / name for name in ("report.csv", "report.xlsx", "report.json", "report.md", "failed_dois.csv")]
zip_paths = zip_downloads_split(
    OUTPUT_DIR,
    n_splits=N_SPLITS,
    zip_name=OUTPUT_DIR.name,
    dois=report["doi"].tolist(),
    out_dir=OUTPUT_DIR.parent,
    extra_files=[f for f in report_files if f.exists()],
)
for zip_path in zip_paths:
    print(f"{zip_path.resolve()} ({zip_path.stat().st_size / 1_048_576:.1f} MB)")

## 7. (Optional) Retry the failed DOIs

Sci-Hub mirrors rotate, so a second attempt later sometimes finds more papers. Recovered PDFs are moved into `OUTPUT_DIR` and marked as downloaded. Re-run sections 5 and 6 afterwards to refresh the report and the ZIP.

In [ ]:
if len(failed):
    retry_dir = OUTPUT_DIR / "retry"
    logging.getLogger().addHandler(tracker)
    try:
        retry_df = run_bulk_download(failed["doi"].tolist(), download_dir=retry_dir, use_browser=USE_BROWSER)
    finally:
        logging.getLogger().removeHandler(tracker)

    for pdf in retry_dir.glob("*.pdf"):
        pdf.replace(OUTPUT_DIR / pdf.name)
    recovered = set(retry_df.loc[retry_df["success"], "doi"])
    results_df.loc[results_df["doi"].isin(recovered), "success"] = True
    print(f"Recovered {len(recovered)}/{len(retry_df)} on retry")
else:
    print("Nothing to retry")